# Expanded LIBERO-PRO analysis (13-suite final cohort)

Zero-GPU analysis of the completed workers 0–5 experiment (`pro-16suite-k5-steps34-v1`). It snapshots only the validated final 13-suite cohort, recreates the success/refinement summaries, reproduces the old notebook's fixed uncertainty-window sweep, and tests whether observed-arm consecutive uncertainty contraction predicts a matched failure-to-success correction.

The contraction analysis uses `pnp_action_vectors.u_iter`; full `a_hat` vectors are neither loaded nor required.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Refresh the Supabase snapshot and run the validated report

In [ ]:
from analysis.run_analysis import main
from pnp.experiments import PRO_EXPANDED_EXPERIMENT

OUTPUT_ROOT = 'analysis_outputs'
main(['pro-expanded', '--experiment', PRO_EXPANDED_EXPERIMENT,
      '--output-root', OUTPUT_ROOT, '--refresh'])

## 3. Tables

Positive contraction scores mean disagreement decreases over the four consecutive perturbation pairs. `corrected` means the observed/no-op rollout failed and the paired refine-last rollout succeeded. All optimal-window results are exploratory: the same outcomes are used to select and evaluate the window.

In [ ]:
import json
import pandas as pd
from analysis.snapshot import latest_snapshot

snapshot_path = latest_snapshot(OUTPUT_ROOT, PRO_EXPANDED_EXPERIMENT)
tables = snapshot_path / 'tables'
print('snapshot:', snapshot_path)
validation = json.loads((snapshot_path / 'validation.json').read_text())
print({key: validation.get(key) for key in (
    'n_snapshot_rollouts', 'n_rollouts', 'n_ignored_rollouts',
    'n_identities', 'n_suites', 'control_complete')})
for warning in validation.get('warnings', []): print('WARNING:', warning)

print('Success and paired transitions')
display(pd.read_csv(tables / 'pro_success_by_suite.csv'))
display(pd.read_csv(tables / 'pro_paired_by_suite.csv'))

print('Consecutive uncertainty contraction')
summary = pd.read_csv(tables / 'expanded_contraction_summary.csv')
display(summary[summary.metric.isin([
    'contraction_within_suite_rank', 'contraction_normalized_slope'
])])
display(pd.read_csv(tables / 'expanded_contraction_trend.csv'))

print('Uncertainty as a failure detector')
display(pd.read_csv(tables / 'pro_detector_summary.csv'))
display(pd.read_csv(tables / 'pro_detector_by_suite.csv'))

print('Dimensional and subspace isolation (mean within-suite failure AUC)')
display(pd.read_csv(tables / 'expanded_dimensional_isolation.csv'))

print('Paired refinement effect by observed-uncertainty decile')
display(pd.read_csv(tables / 'expanded_refinement_effect_by_uncertainty_bin.csv'))

print('Exploratory SR change (percentage points) for every eligible fixed-grid window')
window_sweep = pd.read_csv(tables / 'expanded_uncertainty_window_sweep.csv')
sr_change_table = window_sweep.pivot(index='lower', columns='upper', values='delta_pp')
display(sr_change_table.round(2))

print('Top and bottom windows (at least 20 selected identities; in-sample exploratory)')
display(pd.read_csv(tables / 'expanded_uncertainty_window_extrema.csv'))
print('Selected optimal window')
display(pd.read_csv(tables / 'expanded_uncertainty_optimal_window.csv'))
print('Thresholded selective-policy SR on all 13 suites')
display(pd.read_csv(tables / 'expanded_uncertainty_optimal_by_suite.csv'))

## 4. Figures

For the contraction trend, positive x means consecutive uncertainty decreases across perturbations; negative x means it increases. The ROC/AUC panel asks the same question without choosing bins: AUC 0.5 is no predictive relationship, above 0.5 means stronger contraction predicts correction, and below 0.5 means the opposite.

The per-dimension and subspace plots reproduce the useful isolation analysis from the old notebook. The old first-prediction PCA/isotropy plot is unavailable because these workers ran with `compute_multimodal=False`, so those samples were never collected.

In [ ]:
from IPython.display import Image, display
for name in ('expanded_pro_success_by_suite',
             'expanded_position_strength',
             'expanded_task_success_and_lift',
             'expanded_pro_paired_transitions',
             'expanded_contraction_profiles',
             'expanded_contraction_correction_trend',
             'expanded_contraction_correction_roc',
             'expanded_uncertainty_failure_auc_by_suite',
             'expanded_uncertainty_failure_roc',
             'expanded_uncertainty_per_dimension_auc',
             'expanded_uncertainty_subspace_auc',
             'expanded_uncertainty_by_outcome',
             'expanded_uncertainty_over_time',
             'expanded_refinement_effect_by_uncertainty',
             'expanded_optimal_window_success_by_suite',
             'expanded_uncertainty_window_sweep'):
    print(name)
    display(Image(filename=str(snapshot_path / 'figures' / f'{name}.png')))